# Lab 3 · Live data you can trust

**In class:** Part A, guided, on the World Bank API · **At home:** Part B, the same steps on an API you might use for your project
**How it fits:** this optional notebook shows each pipeline step one at a time. Use it if you want to see what happens inside `pipeline.py`. Labs 3 + 4 are due **Sunday, November 22**.

Tonight's four skills:
1. **Fetch** live data from an API, page by page
2. **Flatten** the response into a table and load it into DuckDB
3. **Check** it against five rules that define "ready", and stop if it fails
4. **Share** the validated table in the cloud, on MotherDuck

Run cells top to bottom with **Shift+Enter**.

---
# Part A · Guided: urban population from the World Bank

The question: **what share of each country's people live in cities, and how is that changing?** The World Bank publishes this as indicator `SP.URB.TOTL.IN.ZS`, "Urban population (% of total population)".

## A1. One request

An API is a web address that returns data instead of a web page. The address has three parts:

- **Base:** `https://api.worldbank.org/v2`
- **Endpoint** (what you're asking for): `/country/all/indicator/SP.URB.TOTL.IN.ZS`
- **Parameters** (options, after the `?`): `format=json`, which years, how many rows per page

In [18]:
import time
from pathlib import Path
import requests
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

BASE = "https://api.worldbank.org/v2"
INDICATOR = "SP.URB.TOTL.IN.ZS"            # urban population, % of total
URL = f"{BASE}/country/all/indicator/{INDICATOR}"
PARAMS = {"format": "json", "date": "2015:2023", "per_page": 1000}

response = requests.get(URL, params={**PARAMS, "page": 1}, timeout=90)
print("status:", response.status_code)    # 200 means OK
response.raise_for_status()               # stop here if the request failed

meta, records = response.json()           # the World Bank returns [page info, rows]
meta

status: 200


{'page': 1,
 'pages': 3,
 'per_page': 1000,
 'total': 2394,
 'sourceid': '2',
 'lastupdated': '2026-11-11'}

`meta` is **information about the data**: which page this is, how many pages there are, and the total number of rows. One record looks like this:

In [19]:
records[0]

{'indicator': {'id': 'SP.URB.TOTL.IN.ZS',
  'value': 'Urban population (% of total population)'},
 'country': {'id': 'ZH', 'value': 'Africa Eastern and Southern'},
 'countryiso3code': 'AFE',
 'date': '2023',
 'value': 37.7723012408349,
 'unit': '',
 'obs_status': '',
 'decimal': 0}

**Notice:** the record is **nested**, like last week's document example. `country` is a little object inside the row, with its own `id` and `value`. We'll flatten it into plain columns.

## A2. Fetch every page, politely

APIs hand out data in pages. We ask for page 1, 2, 3… until we reach the last page, pausing briefly between requests so we don't overload the server (that's what **rate limits** protect against).

In [20]:
def fetch_all_pages(url, params, pause=0.5):
    rows, page = [], 1
    while True:
        r = requests.get(url, params={**params, "page": page}, timeout=90)
        r.raise_for_status()
        meta, records = r.json()
        rows.extend(records or [])
        print(f"page {meta['page']} of {meta['pages']}: {len(records or [])} rows")
        if page >= meta["pages"]:
            return rows, meta
        page += 1
        time.sleep(pause)                 # be polite between requests

raw_rows, meta = fetch_all_pages(URL, {**PARAMS, "per_page": 500})
print("rows received:", len(raw_rows), " rows the API says exist:", meta["total"])

page 1 of 5: 500 rows
page 2 of 5: 500 rows
page 3 of 5: 500 rows
page 4 of 5: 500 rows
page 5 of 5: 394 rows
rows received: 2394  rows the API says exist: 2394


**Rows in should equal rows out**, again. If those two numbers differ, a page went missing.

## A3. Flatten into a table

`pd.json_normalize` turns nested records into columns: `country` → `{id, value}` becomes `country.id` and `country.value`. Then we keep and rename what we need.

In [21]:
urban = pd.json_normalize(raw_rows)
urban = urban.rename(columns={
    "countryiso3code": "country_code",
    "country.value": "country",
    "date": "year",
    "value": "urban_pct",
})[["country_code", "country", "year", "urban_pct"]]

urban["year"] = urban["year"].astype(int)
urban.head()

,country_code,country,year,urban_pct
0,AFE,Africa Eastern and Southern,2023,37.772301
1,AFE,Africa Eastern and Southern,2022,37.360578
2,AFE,Africa Eastern and Southern,2021,36.908543
3,AFE,Africa Eastern and Southern,2020,36.488322
4,AFE,Africa Eastern and Southern,2019,36.097331


## A4. Countries vs. regions: a join from last week

The World Bank mixes **regions** ("World", "Sub-Saharan Africa", "High income") in with countries. If we average across all rows, regions get counted as if they were countries. A second endpoint lists every code with its region, and regions are labelled `Aggregates`. Join, then keep real countries only.

In [22]:
countries_raw, _ = fetch_all_pages(f"{BASE}/country", {"format": "json", "per_page": 100})
countries = pd.json_normalize(countries_raw).rename(columns={
    "id": "country_code", "region.value": "region", "incomeLevel.value": "income_level",
})[["country_code", "region", "income_level"]]

print("codes in the lookup:", len(countries))
countries["region"].value_counts().head(8)

page 1 of 3: 100 rows
page 2 of 3: 100 rows
page 3 of 3: 96 rows
codes in the lookup: 296


region
Aggregates                                           79
Europe & Central Asia                                58
Sub-Saharan Africa                                   48
Latin America & Caribbean                            42
East Asia & Pacific                                  37
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                                         3
Name: count, dtype: int64

In [23]:
con = duckdb.connect(str(ROOT / "lab3.duckdb"))
con.sql("CREATE OR REPLACE TABLE urban_raw AS SELECT * FROM urban")
con.sql("CREATE OR REPLACE TABLE countries AS SELECT * FROM countries")

con.sql("""
    CREATE OR REPLACE TABLE urban_clean AS
    SELECT u.country_code, u.country, c.region, c.income_level, u.year, u.urban_pct
    FROM urban_raw u
    JOIN countries c ON u.country_code = c.country_code
    WHERE trim(c.region) <> 'Aggregates'   -- trim: some region names end in a space
""")

con.sql("""
    SELECT (SELECT count(*) FROM urban_raw)   AS rows_from_api,
           (SELECT count(*) FROM urban_clean) AS rows_for_countries
""")

┌───────────────┬────────────────────┐
│ rows_from_api │ rows_for_countries │
│     int64     │       int64        │
├───────────────┼────────────────────┤
│          2394 │               1953 │
└───────────────┴────────────────────┘

**What happened to the rest?** Fewer rows is expected here: the regions were removed on purpose. That's the difference between losing rows by accident (last week) and dropping them by decision (this week). Write the decision down.
The original dataset had 2,394 rows. After joining the country information table and filtering out aggregates, 1,953 rows remained. We removed regional aggregates because they are not individual countries. This helps avoid double counting and makes the analysis more accurate. The rows were removed intentionally, not lost during data collection.

## A5. Five checks that define "ready"

Before anyone uses this table, it has to pass five rules. Each check is a SQL query that counts **problem rows**; zero problems means pass.

In [24]:
class ValidationError(Exception):
    pass

def run_checks(con, table, checks):
    """Run each check; print PASS/FAIL; stop loudly if any fail."""
    failures = []
    for name, sql in checks:
        problems = con.sql(sql.format(t=table)).fetchone()[0]
        status = "PASS" if problems == 0 else "FAIL"
        print(f"{status}  {name}  ({problems} problem rows)")
        if problems:
            failures.append(name)
    if failures:
        raise ValidationError(f"{table} is NOT ready: {failures}")
    print(f"\n{table} is ready.")

CHECKS = [
    # 1. Completeness of the download: did every page arrive?
    ("all pages arrived",
     f"SELECT abs((SELECT count(*) FROM urban_raw) - {meta['total']})"),
    # 2. Keys: every row has a country and a year, and no country-year appears twice
    ("keys present and unique",
     "SELECT count(*) FROM (SELECT country_code, year, count(*) n FROM {t} "
     "GROUP BY ALL HAVING n > 1 OR country_code IS NULL OR year IS NULL)"),
    # 3. Validity: a percentage must be between 0 and 100
    ("urban_pct between 0 and 100",
     "SELECT count(*) FROM {t} WHERE urban_pct < 0 OR urban_pct > 100"),
    # 4. Completeness of values: no more than 25% of values missing
    ("under 25% of values missing",
     "SELECT CASE WHEN avg(CASE WHEN urban_pct IS NULL THEN 1 ELSE 0 END) > 0.25 THEN 1 ELSE 0 END FROM {t}"),
    # 5. Freshness: there is data for 2022 or later
    ("has data from 2022 or later",
     "SELECT CASE WHEN max(year) FILTER (WHERE urban_pct IS NOT NULL) >= 2022 THEN 0 ELSE 1 END FROM {t}"),
]

run_checks(con, "urban_clean", CHECKS)

PASS  all pages arrived  (0 problem rows)
PASS  keys present and unique  (0 problem rows)
PASS  urban_pct between 0 and 100  (0 problem rows)
PASS  under 25% of values missing  (0 problem rows)
PASS  has data from 2022 or later  (0 problem rows)

urban_clean is ready.


## A6. What failure looks like

Real data breaks. Here we copy the table and plant one impossible value: a country that's 150% urban. Watch the checks catch it and **stop**.

In [25]:
con.sql("CREATE OR REPLACE TABLE urban_broken AS SELECT * FROM urban_clean")
con.sql("UPDATE urban_broken SET urban_pct = 150 WHERE country_code = 'KEN' AND year = 2020")

try:
    run_checks(con, "urban_broken", CHECKS)
except ValidationError as e:
    print("\nSTOPPED:", e)
    print("Nothing was written to the cloud. The last good table is still there.")

PASS  all pages arrived  (0 problem rows)
PASS  keys present and unique  (0 problem rows)
FAIL  urban_pct between 0 and 100  (1 problem rows)
PASS  under 25% of values missing  (0 problem rows)
PASS  has data from 2022 or later  (0 problem rows)

STOPPED: urban_broken is NOT ready: ['urban_pct between 0 and 100']
Nothing was written to the cloud. The last good table is still there.


That's what **failing loudly** means: the bad data never reaches anyone downstream, and the message says exactly which rule broke. In Session 4 this same function guards an automatic, daily pipeline.

## A7. Share it: write to MotherDuck

Your `lab3.duckdb` file lives only in this Codespace. **MotherDuck** is DuckDB in the cloud: a table there can be reached by a scheduled job or a web app.

**Before running this cell**, you need your MotherDuck token saved as a Codespaces secret named `MOTHERDUCK_TOKEN` (we do this together in class; steps are in `labs/lab3.md`). The code never contains the token: DuckDB reads it from the secret automatically.

In [26]:
import os

# GitHub stores secret names in capitals; DuckDB looks for either spelling. This line covers both.
token = os.environ.get("MOTHERDUCK_TOKEN") or os.environ.get("motherduck_token")
if token:
    os.environ["motherduck_token"] = token

if not token:
    print("No MotherDuck token found. Follow 'Set up your MotherDuck token' in labs/lab3.md,")
    print("then restart your Codespace and run this cell again.")
else:
    run_checks(con, "urban_clean", CHECKS)          # never publish unchecked data
    to_share = con.sql("SELECT * FROM urban_clean").df()

    md = duckdb.connect("md:")                       # 'md:' means MotherDuck
    md.sql("CREATE DATABASE IF NOT EXISTS course")
    md.register("to_share", to_share)
    md.sql("CREATE OR REPLACE TABLE course.main.urban_population AS SELECT * FROM to_share")
    print(md.sql("SELECT count(*) AS rows_in_cloud FROM course.main.urban_population"))

PASS  all pages arrived  (0 problem rows)
PASS  keys present and unique  (0 problem rows)
PASS  urban_pct between 0 and 100  (0 problem rows)
PASS  under 25% of values missing  (0 problem rows)
PASS  has data from 2022 or later  (0 problem rows)

urban_clean is ready.


┌───────────────┐
│ rows_in_cloud │
│     int64     │
├───────────────┤
│          1953 │
└───────────────┘



Open **app.motherduck.com** in your browser: you should see a database called `course` with a table `urban_population`.

## A8. Use it

Which countries urbanized fastest between 2015 and 2023? This query runs on the validated data.

In [27]:
con.sql("""
    SELECT country, region,
           max(urban_pct) FILTER (WHERE year = 2015) AS pct_2015,
           max(urban_pct) FILTER (WHERE year = 2023) AS pct_2023,
           round(pct_2023 - pct_2015, 1)              AS change
    FROM urban_clean
    GROUP BY country, region
    HAVING pct_2015 IS NOT NULL AND pct_2023 IS NOT NULL
    ORDER BY change DESC
    LIMIT 10
""")

┌────────────────────┬────────────────────────────┬──────────────────┬──────────────────┬────────┐
│      country       │           region           │     pct_2015     │     pct_2023     │ change │
│      varchar       │          varchar           │      double      │      double      │ double │
├────────────────────┼────────────────────────────┼──────────────────┼──────────────────┼────────┤
│ Thailand           │ East Asia & Pacific        │ 46.9521043776171 │ 60.9341926574707 │   14.0 │
│ Rwanda             │ Sub-Saharan Africa         │ 18.1953149858416 │ 28.8555310645477 │   10.7 │
│ Kiribati           │ East Asia & Pacific        │ 50.9541970402064 │ 61.5244676194408 │   10.6 │
│ Cambodia           │ East Asia & Pacific        │ 30.0669222549819 │ 40.6290788072508 │   10.6 │
│ Somalia, Fed. Rep. │ Sub-Saharan Africa         │ 43.8618107408479 │ 54.0714435506882 │   10.2 │
│ Switzerland        │ Europe & Central Asia      │ 75.6044960143169 │ 85.1854212679815 │    9.6 │
│ Kosovo  

---
# Part B · At home: an API you might use for your project

Pick a public API you're considering for your final project (your dataset choice is due **November 15**, with this lab). Many of the portals from Lab 1 have one:

- **NYC Open Data** (and most city portals): every dataset page has an **API** button. URLs look like `https://data.cityofnewyork.us/resource/xxxx-xxxx.json`
- **World Bank:** any indicator, using the same code as Part A
- Others: data.gov, Census, transit agencies. Prefer APIs that **don't need a key**.

## B1. One request

In [28]:
import requests
MY_URL = "https://data.cityofnewyork.us/resource/rbx6-tga4.json"   # <- your API's address
MY_PARAMS = {
    "$select": "work_permit,sequence_number,filing_reason,work_type,borough,bin,community_board,issued_date,expired_date,permit_status",
    "$where": "upper(borough)='MANHATTAN' AND work_type='Sidewalk Shed' AND issued_date IS NOT NULL",
    "$order": "issued_date DESC,:id ASC",
    "$limit": 2
    }                                                     # <- any filters, e.g. {"borough": "BROOKLYN"}

r = requests.get(MY_URL, params=MY_PARAMS, timeout=90)
print("status:", r.status_code)
r.raise_for_status()
sample = r.json()
sample[:2] if isinstance(sample, list) else sample

status: 200


[{'work_permit': 'M01412019-S1-SH',
  'sequence_number': '1',
  'filing_reason': 'Initial Permit',
  'work_type': 'Sidewalk Shed',
  'borough': 'Manhattan',
  'bin': '1084726',
  'community_board': '108',
  'issued_date': '2026-11-20T00:00:00.000',
  'expired_date': '2027-01-01T00:00:00.000',
  'permit_status': 'Permit Issued'},
 {'work_permit': 'M00931319-I1-SH',
  'sequence_number': '5',
  'filing_reason': 'Renewal Permit with Changes',
  'work_type': 'Sidewalk Shed',
  'borough': 'Manhattan',
  'bin': '1045154',
  'community_board': '108',
  'issued_date': '2026-11-20T00:00:00.000',
  'expired_date': '2027-02-17T00:00:00.000',
  'permit_status': 'Permit Issued'}]

**Describe the response** (double-click to edit): Is it a list of records, or page info plus records (like the World Bank)? Which fields are nested? What will one row of your table be (the grain)?
The API returns a list of records, not page information plus records like the World Bank API. The fields are flat, with no nested objects. Each row represents one sidewalk shed permit record in Manhattan, including initial permits and renewals. One row does not necessarily represent one unique physical sidewalk shed.

## B2. Fetch at least two pages

Pick the pattern your API uses. **NYC Open Data and other Socrata portals** use `$limit` and `$offset`; the World Bank uses `page`.

In [29]:
import requests
import time
def fetch_socrata(url, params=None, page_size=1000, max_pages=5, pause=0.5):
    """NYC Open Data style: ask for page_size rows at a time, skipping ahead with $offset."""
    rows = []
    for i in range(max_pages):
        p = {**(params or {}), "$limit": page_size, "$offset": i * page_size}
        r = requests.get(url, params=p, timeout=90)
        r.raise_for_status()
        batch = r.json()
        print(f"page {i + 1}: {len(batch)} rows")
        rows.extend(batch)
        if len(batch) < page_size:    # a short page means we've reached the end
            break
        time.sleep(pause)
    return rows

my_rows = fetch_socrata(MY_URL, MY_PARAMS)
# World Bank instead? Use:  my_rows, my_meta = fetch_all_pages(MY_URL, {"format": "json", ...})
print("rows:", len(my_rows))

page 1: 1000 rows
page 2: 1000 rows
page 3: 1000 rows
page 4: 1000 rows
page 5: 1000 rows
rows: 5000


## B3. Flatten, clean, load

In [30]:
mine = pd.json_normalize(my_rows)
print(mine.columns.tolist())

# Clean: rename, keep what you need, fix types. Comment each step with why.
# mine = mine.rename(columns={...})[[...]]
# mine["created_date"] = pd.to_datetime(mine["created_date"])

con.sql("CREATE OR REPLACE TABLE my_data AS SELECT * FROM mine")
con.sql("DESCRIBE my_data")

['work_permit', 'sequence_number', 'filing_reason', 'work_type', 'borough', 'bin', 'community_board', 'issued_date', 'expired_date', 'permit_status']


┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ work_permit     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sequence_number │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ filing_reason   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ work_type       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ borough         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bin             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ community_board │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ issued_date     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ expired_date    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ permit_status   │ VARCHAR     │ YES 

In [31]:

import pandas as pd

# Flatten the API records into a table.
mine = pd.json_normalize(my_rows)
print(mine.columns.tolist())

# Keep the fields needed for my sidewalk shed project.
mine = mine[[
    "work_permit",
    "sequence_number",
    "filing_reason",
    "work_type",
    "borough",
    "bin",
    "community_board",
    "issued_date",
    "expired_date",
    "permit_status"
]].copy()

# The original column names are already clear, so no renaming is needed.

# Convert sequence numbers to integers to study permit renewals.
mine["sequence_number"] = pd.to_numeric(
    mine["sequence_number"], errors="coerce"
).astype("Int64")

# Convert dates so I can calculate permit age later.
mine["issued_date"] = pd.to_datetime(
    mine["issued_date"], errors="coerce"
)
mine["expired_date"] = pd.to_datetime(
    mine["expired_date"], errors="coerce"
)

# Keep identifying codes as strings, not numeric values.
for col in ["work_permit", "bin", "community_board"]:
    mine[col] = mine[col].astype("string").str.strip()

# Keep missing values for data quality checks in B4.
# Do not remove renewals because they are important for my research.

# Load the cleaned data into DuckDB.
con.sql("CREATE OR REPLACE TABLE my_data AS SELECT * FROM mine")

# Check the table structure and number of records.
print(con.sql("DESCRIBE my_data"))
print(con.sql("SELECT COUNT(*) AS total_rows FROM my_data"))


['work_permit', 'sequence_number', 'filing_reason', 'work_type', 'borough', 'bin', 'community_board', 'issued_date', 'expired_date', 'permit_status']
┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ work_permit     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sequence_number │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ filing_reason   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ work_type       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ borough         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bin             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ community_board │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ issued_date     │ TIMESTAMP   │ YE

## B4. Your five checks

Write five checks that define "ready" for **your** data. Each is a SQL query that returns the number of problem rows (`{t}` stands for the table name). Aim for one of each kind: completeness, keys, validity, allowed values, freshness.

In [32]:

MY_CHECKS = [
    # 1. Completeness: important fields should not be missing.
    ("1. completeness: required fields",
     """SELECT count(*) FROM {t}
        WHERE work_permit IS NULL
           OR trim(work_permit) = ''
           OR borough IS NULL
           OR work_type IS NULL"""),

    # 2. Keys: permit records should be unique.
    ("2. keys unique",
     """SELECT count(*) FROM (
        SELECT work_permit, sequence_number, count(*) AS n
        FROM {t}
        GROUP BY work_permit, sequence_number
        HAVING n > 1
           OR work_permit IS NULL
           OR sequence_number IS NULL
     )"""),

    # 3. Validity: dates and sequence numbers should be valid.
    ("3. valid dates and sequence",
     """SELECT count(*) FROM {t}
        WHERE expired_date < issued_date
           OR sequence_number < 1"""),

    # 4. Allowed values: only Manhattan sidewalk shed permits.
    ("4. allowed values",
     """SELECT count(*) FROM {t}
        WHERE coalesce(upper(trim(borough)), '') <> 'MANHATTAN'
           OR coalesce(work_type, '') <> 'Sidewalk Shed'"""),

    # 5. Freshness: at least one recent permit.
    ("5. freshness",
     """SELECT CASE
        WHEN max(issued_date) >= CURRENT_DATE - INTERVAL '30 days'
        THEN 0 ELSE 1 END
        FROM {t}""")
]

run_checks(con, "my_data", MY_CHECKS)


PASS  1. completeness: required fields  (0 problem rows)
FAIL  2. keys unique  (4 problem rows)
PASS  3. valid dates and sequence  (0 problem rows)
PASS  4. allowed values  (0 problem rows)
PASS  5. freshness  (0 problem rows)


ValidationError: my_data is NOT ready: ['2. keys unique']

In [ ]:

con.sql("""
SELECT
    work_permit,
    sequence_number,
    COUNT(*) AS record_count
FROM my_data
GROUP BY work_permit, sequence_number
HAVING COUNT(*) > 1
    OR work_permit IS NULL
    OR sequence_number IS NULL
ORDER BY record_count DESC
""").df()


,work_permit,sequence_number,record_count
0,M01089754-I1-SH,5,2
1,M00874930-S2-SH,4,2
2,M01216281-I1-SH,3,2
3,M00874930-S2-SH,3,2


In [ ]:

con.sql("""
SELECT *,
    COUNT(*) OVER (
        PARTITION BY work_permit, sequence_number
    ) AS same_key_count
FROM my_data
QUALIFY same_key_count > 1
    OR work_permit IS NULL
    OR sequence_number IS NULL
ORDER BY work_permit, sequence_number, issued_date
""").df()


,work_permit,sequence_number,filing_reason,work_type,borough,bin,community_board,issued_date,expired_date,permit_status,same_key_count
0,M00874930-S2-SH,3,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1091905,101,2026-07-08,2026-10-06 04:00:00,Permit Issued,2
1,M00874930-S2-SH,3,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1091905,101,2026-07-08,2026-10-06 04:00:00,Permit Issued,2
2,M00874930-S2-SH,4,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1091905,101,2026-10-06,2027-01-04 00:00:00,Permit Issued,2
3,M00874930-S2-SH,4,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1091905,101,2026-10-06,2027-01-04 00:00:00,Permit Issued,2
4,M01089754-I1-SH,5,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1024811,105,2026-08-24,2026-11-22 05:00:00,Permit Issued,2
5,M01089754-I1-SH,5,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1024811,105,2026-11-10,2027-02-08 00:00:00,Permit Issued,2
6,M01216281-I1-SH,3,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1046971,108,2026-06-30,2026-09-28 04:00:00,Permit Issued,2
7,M01216281-I1-SH,3,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1046971,108,2026-09-28,2026-12-27 00:00:00,Permit Issued,2


In [ ]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM my_data)
    -
    (SELECT COUNT(*) FROM
        (SELECT DISTINCT * FROM my_data))
    AS exact_duplicate_rows
""").df()

,exact_duplicate_rows
0,2


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE my_data_dedup AS
SELECT DISTINCT *
FROM my_data
""")

con.sql("""
SELECT COUNT(*) AS remaining_rows
FROM my_data_dedup
""").df()

,remaining_rows
0,4998


In [ ]:
run_checks(con, "my_data_dedup", MY_CHECKS)

PASS  1. completeness: required fields  (0 problem rows)
FAIL  2. keys unique  (2 problem rows)
PASS  3. valid dates and sequence  (0 problem rows)
PASS  4. allowed values  (0 problem rows)
PASS  5. freshness  (0 problem rows)


ValidationError: my_data_dedup is NOT ready: ['2. keys unique']

In [ ]:

con.sql("""
SELECT
    work_permit,
    sequence_number,
    COUNT(*) AS record_count
FROM my_data_dedup
GROUP BY work_permit, sequence_number
HAVING COUNT(*) > 1
    OR work_permit IS NULL
    OR sequence_number IS NULL
ORDER BY record_count DESC
""").df()


,work_permit,sequence_number,record_count
0,M01216281-I1-SH,3,2
1,M01089754-I1-SH,5,2


In [ ]:

duplicates = con.sql("""
SELECT *
FROM my_data_dedup
WHERE (work_permit, sequence_number) IN (
    SELECT work_permit, sequence_number
    FROM my_data_dedup
    GROUP BY work_permit, sequence_number
    HAVING COUNT(*) > 1
)
ORDER BY work_permit, sequence_number, issued_date
""").df()

display(duplicates)


,work_permit,sequence_number,filing_reason,work_type,borough,bin,community_board,issued_date,expired_date,permit_status
0,M01089754-I1-SH,5,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1024811,105,2026-08-24,2026-11-22 05:00:00,Permit Issued
1,M01089754-I1-SH,5,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1024811,105,2026-11-10,2027-02-08 00:00:00,Permit Issued
2,M01216281-I1-SH,3,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1046971,108,2026-06-30,2026-09-28 04:00:00,Permit Issued
3,M01216281-I1-SH,3,Renewal Permit Without Changes,Sidewalk Shed,Manhattan,1046971,108,2026-09-28,2026-12-27 00:00:00,Permit Issued


In [ ]:

con.sql("""
SELECT
    work_permit,
    sequence_number,
    issued_date,
    COUNT(*) AS n
FROM my_data_dedup
GROUP BY work_permit, sequence_number, issued_date
HAVING COUNT(*) > 1
    OR work_permit IS NULL
    OR sequence_number IS NULL
    OR issued_date IS NULL
""").df()


,work_permit,sequence_number,issued_date,n


In [ ]:

MY_CHECKS[1] = (
    "2. keys present and unique",
    """SELECT count(*) FROM (
        SELECT work_permit, sequence_number, issued_date,
               count(*) AS n
        FROM {t}
        GROUP BY work_permit, sequence_number, issued_date
        HAVING n > 1
            OR work_permit IS NULL
            OR sequence_number IS NULL
            OR issued_date IS NULL
    )"""
)

run_checks(con, "my_data_dedup", MY_CHECKS)


PASS  1. completeness: required fields  (0 problem rows)
PASS  2. keys present and unique  (0 problem rows)
PASS  3. valid dates and sequence  (0 problem rows)
PASS  4. allowed values  (0 problem rows)
PASS  5. freshness  (0 problem rows)

my_data_dedup is ready.


**Why these five?** For each check, one sentence: what would go wrong downstream if it failed?

1. Completeness: If important fields are missing, I may not be able to identify and analyze some sidewalk shed permits.
2. Keys: If records are duplicated, I may overcount permits and get inaccurate results for different community districts.
3. Validity: If permit dates or sequence numbers are incorrect, my analysis of permit age may be inaccurate.
4. Allowed values: If the data includes other boroughs or work types, my analysis of Manhattan sidewalk sheds may be incorrect.
5. Freshness: If the data is outdated, it may not reflect recent sidewalk shed permits and changes.

## B5. Write it to MotherDuck

In [ ]:

import os
import duckdb

# Name the cloud table for my sidewalk shed project.
MY_TABLE = "manhattan_sidewalk_sheds"

# Only publish the table that passed all five checks.
run_checks(con, "my_data_dedup", MY_CHECKS)

# Read the MotherDuck token from Codespaces secrets.
os.environ.setdefault(
    "motherduck_token",
    os.environ.get("MOTHERDUCK_TOKEN", "")
)

# Connect to MotherDuck.
md = duckdb.connect("md:")

# Create the database if it does not exist.
md.sql("CREATE DATABASE IF NOT EXISTS course")

# Prepare the validated data for upload.
to_share = con.sql("SELECT * FROM my_data_dedup").df()
md.register("to_share", to_share)

# Write the validated table to MotherDuck.
md.sql(
    f"CREATE OR REPLACE TABLE course.main.{MY_TABLE} AS SELECT * FROM to_share"
)

# Verify the number of rows in the cloud.
print(
    md.sql(
        f"SELECT count(*) AS rows_in_cloud FROM course.main.{MY_TABLE}"
    )
)


PASS  1. completeness: required fields  (0 problem rows)
PASS  2. keys present and unique  (0 problem rows)
PASS  3. valid dates and sequence  (0 problem rows)
PASS  4. allowed values  (0 problem rows)
PASS  5. freshness  (0 problem rows)

my_data_dedup is ready.
┌───────────────┐
│ rows_in_cloud │
│     int64     │
├───────────────┤
│          4998 │
└───────────────┘



In [ ]:
print(md.sql("SHOW TABLES FROM course.main"))

┌──────────────────┐
│       name       │
│     varchar      │
├──────────────────┤
│ urban_population │
└──────────────────┘



In [33]:

MY_CHECKS[1] = (
    "2. keys present and unique",
    """SELECT count(*) FROM (
        SELECT work_permit, sequence_number, issued_date,
               count(*) AS n
        FROM {t}
        GROUP BY work_permit, sequence_number, issued_date
        HAVING n > 1
            OR work_permit IS NULL
            OR sequence_number IS NULL
            OR issued_date IS NULL
    )"""
)

run_checks(con, "my_data_dedup", MY_CHECKS)


PASS  1. completeness: required fields  (0 problem rows)
PASS  2. keys present and unique  (0 problem rows)
PASS  3. valid dates and sequence  (0 problem rows)
PASS  4. allowed values  (0 problem rows)
PASS  5. freshness  (0 problem rows)

my_data_dedup is ready.


In [34]:

import duckdb

# Check the cleaned data before uploading.
run_checks(con, "my_data_dedup", MY_CHECKS)

# Prepare the validated data.
to_share = con.sql(
    "SELECT * FROM my_data_dedup"
).df()

print("Local rows:", len(to_share))

# Connect to MotherDuck.
md = duckdb.connect("md:")

# Register the DataFrame.
md.register("to_share", to_share)

# Upload the table.
md.execute("""
    CREATE OR REPLACE TABLE
    course.main.manhattan_sidewalk_sheds
    AS SELECT * FROM to_share
""")

md.commit()

# Verify the uploaded table.
print(md.sql("""
    SELECT COUNT(*) AS rows_in_cloud
    FROM course.main.manhattan_sidewalk_sheds
"""))

print(md.sql("SHOW TABLES FROM course.main"))


PASS  1. completeness: required fields  (0 problem rows)
PASS  2. keys present and unique  (0 problem rows)
PASS  3. valid dates and sequence  (0 problem rows)
PASS  4. allowed values  (0 problem rows)
PASS  5. freshness  (0 problem rows)

my_data_dedup is ready.
Local rows: 4998
┌───────────────┐
│ rows_in_cloud │
│     int64     │
├───────────────┤
│          4998 │
└───────────────┘

┌──────────────────────────┐
│           name           │
│         varchar          │
├──────────────────────────┤
│ manhattan_sidewalk_sheds │
│ urban_population         │
└──────────────────────────┘



## B6. Next: put it in your pipeline

You don't need to copy any Python. Your pipeline is set up in one settings file, `project.toml`:

- the API address and columns go in **[source]**
- your checks from B4 go in **[[checks]]** (the same SQL, with `raw` as the table name)
- the table you want your app to use goes in **[[tables]]**

Run `python pipeline.py --preview` in the terminal to see your API's columns. `examples/project_worldbank.toml` shows a World Bank version.

**Never commit your token.** It lives only in your secrets.